# Module 3 & 4 — Phase 5: End-to-End Unified System Integration & Risk Assessment Test Notebook

This notebook tests the Phase 5 unified screening pipeline, advisory risk assessment, and **Module 4 Face Verification** system.

## Objectives:
1. Test `run_screening()` pipeline on passport images and current face photos.
2. Inspect outputs from all 4 modules (Module 1 OCR, Module 2 Database Validation, Module 3 Tampering/ELA/Metadata/EfficientNet-B0, Module 4 Face Verification).
3. Inspect advisory risk assessment scores and decision logic combining all 4 modules.
4. Test JSON and CSV report exports.

In [ ]:
import sys
import json
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.screening_service import run_screening, export_report_json, export_report_csv
from app.services.risk_assessment import assess_risk, RiskConfig

print("Project Root:", PROJECT_ROOT)
print("Phase 5 & Module 4 Integration modules loaded successfully.")

## 1. Run 4-Module Unified Screening on Passport Image + Current Face Photo

In [ ]:
test_passport_path = PROJECT_ROOT / "face_verification" / "passport2.jpg"
test_current_face_path = PROJECT_ROOT / "face_verification" / "current_image.jpg"

if not test_passport_path.exists():
    # Fallback to passport/passport1.jpg if face_verification directory not present
    test_passport_path = PROJECT_ROOT / "passport" / "passport1.jpg"

print(f"Running 4-module unified screening pipeline on {test_passport_path.name}...")
report = run_screening(
    image_input=test_passport_path,
    current_face_image_input=test_current_face_path if test_current_face_path.exists() else None,
    filename=test_passport_path.name,
)

print("\n--- SCREENING REPORT SUMMARY ---")
print(f"Screening ID: {report.get('screening_id')}")
print(f"Timestamp:    {report.get('timestamp')}")
print(f"Filename:     {report.get('filename')}")
print("\nModule Statuses:")
for mod, stat in report.get("module_statuses", {}).items():
    print(f"  - {mod:20s}: {stat}")

## 2. Inspect Module 4 Face Verification & Age Context Output

In [ ]:
face_sec = report.get("face_verification", {})
print(f"Module 4 Status: {face_sec.get('status')}")
face_res = face_sec.get("result")
if face_res:
    print("=== MODULE 4 FACE VERIFICATION FINDINGS ===")
    print(f"Verification Status:      {face_res.get('face_verification_status')}")
    print(f"Policy Case:              {face_res.get('policy_case')}")
    print(f"Raw Distance / Threshold: {face_res.get('raw_distance')} / {face_res.get('model_threshold')}")
    print(f"Derived Similarity Score: {face_res.get('derived_similarity_score')}%")
    print(f"Resolved Issue Date:      {face_res.get('resolved_issue_date')} (Source: {face_res.get('issue_date_source')})")
    print(f"Document Elapsed Age:     {face_res.get('elapsed_years')} years")
    print(f"Manual Review Required:   {face_res.get('manual_review_recommended')}")
    print(f"\nExplanation:\n  {face_res.get('human_readable_explanation')}")
else:
    print(f"Face verification result unavailable: {face_sec.get('error')}")

## 3. Inspect Unified Advisory Risk Assessment Results

In [ ]:
risk_res = (report.get("risk_assessment") or {}).get("result")
if risk_res:
    print("=== UNIFIED ADVISORY RISK ASSESSMENT ===")
    print(f"Risk Level:                {risk_res.get('risk_level')}")
    print(f"Risk Score:                {risk_res.get('risk_score')}")
    print(f"Manual Review Recommended: {risk_res.get('manual_review_recommended')}")
    print("\nRisk Signal Reasons:")
    for reason in risk_res.get("risk_reasons", []):
        print(f"  - {reason}")
    print(f"\nSignals Used:    {risk_res.get('signals_used')}")
    print(f"Signals Missing: {risk_res.get('signals_missing')}")
    print(f"\nDisclaimer:\n  {risk_res.get('disclaimer')}")
else:
    print("Risk assessment not available in report.")

## 4. Test Report Exporting (JSON & CSV)

In [ ]:
output_dir = PROJECT_ROOT / "data" / "reports"
output_dir.mkdir(parents=True, exist_ok=True)

json_out = output_dir / "sample_screening_report_with_face.json"
csv_out  = output_dir / "sample_screening_report_with_face.csv"

export_report_json(report, json_out)
export_report_csv(report, csv_out)

print(f"Exported JSON report to: {json_out}")
print(f"Exported CSV report to:  {csv_out}")

## 5. Batch Screening Test on Sample Passports

In [ ]:
passport_dir = PROJECT_ROOT / "passport"
passport_files = sorted(list(passport_dir.glob("*.jpg"))) + sorted(list(passport_dir.glob("*.png")))

print(f"Found {len(passport_files)} passport files in {passport_dir}")
batch_reports = []

for p_file in passport_files[:3]:  # Test first 3 files
    rep = run_screening(image_input=p_file, filename=p_file.name)
    risk = (rep.get("risk_assessment") or {}).get("result") or {}
    val  = (rep.get("validation") or {}).get("result") or {}
    fv   = (rep.get("face_verification") or {}).get("status")
    
    print(f"File: {p_file.name:15s} | DB Status: {val.get('overall_status', 'N/A'):12s} | Face: {fv:12s} | Risk: {risk.get('risk_level', 'N/A'):8s} | Score: {risk.get('risk_score', 0):.1f}")
    batch_reports.append(rep)

print("\nBatch screening complete.")